![图片描述](seq2seq.jpg)

![图片描述](seq2seq01.jpg)

### Mini-Translation Task

We are performing a tiny translation task:

* **English Input:** `i am happy`
* **Chinese Output:** `我 很 开心` (I am very happy)

**Using the most classic architecture:**

* **Encoder RNN:** Reads the English sentence one word at a time.
* **Decoder RNN:** Generates the Chinese translation one word at a time based on the final hidden state of the Encoder.

In [1]:
import numpy as np

In [2]:
pairs = [
    ("i am happy",      "我 很 开心"),
    ("i am sad",        "我 很 难过"),
    ("you are kind",    "你 很 善良"),
    ("you are smart",   "你 很 聪明"),
    ("he is tall",      "他 很 高"),
    ("he is strong",    "他 很 强壮"),
    ("she is kind",     "她 很 善良"),
    ("she is smart",    "她 很 聪明"),
]

In [3]:
def build_vocab(sentences, add_special_tokens=None):
    vocab = set()
    for s in sentences:
        vocab.update(s.split())
    
    if add_special_tokens:
        vocab.update(add_special_tokens)
    
    vocab = sorted(list(vocab))
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    idx_to_word = {i: w for w, i in word_to_idx.items()}
    return vocab, word_to_idx, idx_to_word

In [4]:
src_sentences = [src for src, tgt in pairs]
tgt_sentences = [tgt for src, tgt in pairs]

src_vocab, src_word_to_idx, src_idx_to_word = build_vocab(src_sentences)
tgt_vocab, tgt_word_to_idx, tgt_idx_to_word = build_vocab(
    tgt_sentences,
    add_special_tokens=["<START>", "<END>"]
)

print("源语言词表:", src_vocab)
print("目标语言词表:", tgt_vocab)

源语言词表: ['am', 'are', 'happy', 'he', 'i', 'is', 'kind', 'sad', 'she', 'smart', 'strong', 'tall', 'you']
目标语言词表: ['<END>', '<START>', '他', '你', '善良', '她', '开心', '强壮', '很', '我', '聪明', '难过', '高']


In [5]:
def one_hot(index, vocab_size):
    x = np.zeros((vocab_size, 1))
    x[index] = 1
    return x

In [6]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / np.sum(e_x, axis=0, keepdims=True)

In [7]:
def initialize_seq2seq_parameters(n_a, n_x_src, n_x_tgt, n_y_tgt):
    np.random.seed(1)
    parameters = {
        "Wax_enc": np.random.randn(n_a, n_x_src) * 0.01,
        "Waa_enc": np.random.randn(n_a, n_a) * 0.01,
        "ba_enc": np.zeros((n_a, 1)),

        "Wax_dec": np.random.randn(n_a, n_x_tgt) * 0.01,
        "Waa_dec": np.random.randn(n_a, n_a) * 0.01,
        "Wya_dec": np.random.randn(n_y_tgt, n_a) * 0.01,
        "ba_dec": np.zeros((n_a, 1)),
        "by": np.zeros((n_y_tgt, 1)),   # 一定要有这一行
    }
    return parameters

In [8]:
def encoder_cell_forward(xt, h_prev, parameters):
    Wax_enc = parameters["Wax_enc"]
    Waa_enc = parameters["Waa_enc"]
    ba_enc  = parameters["ba_enc"]
    
    h_next = np.tanh(np.dot(Wax_enc, xt) + np.dot(Waa_enc, h_prev) + ba_enc)
    cache = (h_next, h_prev, xt)
    return h_next, cache

In [9]:
def encoder_forward(src_indices, h0, parameters, src_vocab_size):
    x, h = {}, {}
    caches = []
    
    h[-1] = np.copy(h0)
    
    for t in range(len(src_indices)):
        x[t] = one_hot(src_indices[t], src_vocab_size)
        h[t], cache = encoder_cell_forward(x[t], h[t-1], parameters)
        caches.append(cache)
    
    return h[len(src_indices)-1], (x, h, caches)

In [10]:
def decoder_cell_forward(yt_prev, s_prev, parameters):
    Wax_dec = parameters["Wax_dec"]
    Waa_dec = parameters["Waa_dec"]
    Wya_dec = parameters["Wya_dec"]
    ba_dec = parameters["ba_dec"]
    by = parameters["by"]

    s_next = np.tanh(np.dot(Wax_dec, yt_prev) + np.dot(Waa_dec, s_prev) + ba_dec)
    y_pred = softmax(np.dot(Wya_dec, s_next) + by)

    cache = (s_next, s_prev, yt_prev)
    return s_next, y_pred, cache

In [11]:
def decoder_forward(tgt_input_indices, tgt_output_indices, s0, parameters, tgt_vocab_size):
    y_in, s, y_pred = {}, {}, {}
    caches = []
    loss = 0.0
    
    s[-1] = np.copy(s0)
    
    for t in range(len(tgt_input_indices)):
        y_in[t] = one_hot(tgt_input_indices[t], tgt_vocab_size)
        s[t], y_pred[t], cache = decoder_cell_forward(y_in[t], s[t-1], parameters)
        caches.append(cache)
        loss += -np.log(y_pred[t][tgt_output_indices[t], 0] + 1e-12)
    
    return loss, (y_in, s, y_pred, caches)

In [12]:
def seq2seq_forward(src_indices, tgt_input_indices, tgt_output_indices,
                    h0, parameters, src_vocab_size, tgt_vocab_size):

    h_enc, enc_cache = encoder_forward(src_indices, h0, parameters, src_vocab_size)
    loss, dec_cache = decoder_forward(
        tgt_input_indices,
        tgt_output_indices,
        h_enc,
        parameters,
        tgt_vocab_size
    )

    cache = {
        "enc_cache": enc_cache,
        "dec_cache": dec_cache,
        "h_enc": h_enc
    }
    return loss, cache

In [13]:
def decoder_cell_backward(dy, ds_next, cache, parameters, gradients):
    s_next, s_prev, yt_prev = cache
    
    Wya_dec = parameters["Wya_dec"]
    Waa_dec = parameters["Waa_dec"]
    
    gradients["dWya_dec"] += np.dot(dy, s_next.T)
    gradients["dby"] += dy
    
    ds = np.dot(Wya_dec.T, dy) + ds_next
    dtanh = (1 - s_next**2) * ds
    
    gradients["dba_dec"] += dtanh
    gradients["dWax_dec"] += np.dot(dtanh, yt_prev.T)
    gradients["dWaa_dec"] += np.dot(dtanh, s_prev.T)
    
    ds_prev = np.dot(Waa_dec.T, dtanh)
    return ds_prev, gradients

In [14]:
def decoder_backward(tgt_output_indices, dec_cache, parameters):
    y_in, s, y_pred, caches = dec_cache
    n_a = parameters["Waa_dec"].shape[0]
    
    gradients = {
        "dWax_dec": np.zeros_like(parameters["Wax_dec"]),
        "dWaa_dec": np.zeros_like(parameters["Waa_dec"]),
        "dWya_dec": np.zeros_like(parameters["Wya_dec"]),
        "dba_dec": np.zeros_like(parameters["ba_dec"]),
        "dby": np.zeros_like(parameters["by"]),
    }
    
    ds_next = np.zeros((n_a, 1))
    
    for t in reversed(range(len(tgt_output_indices))):
        dy = np.copy(y_pred[t])
        dy[tgt_output_indices[t]] -= 1
        ds_next, gradients = decoder_cell_backward(dy, ds_next, caches[t], parameters, gradients)
    
    return ds_next, gradients

In [15]:
def encoder_cell_backward(dh, cache, parameters, gradients):
    h_next, h_prev, xt = cache
    Waa_enc = parameters["Waa_enc"]
    
    dtanh = (1 - h_next**2) * dh
    
    gradients["dba_enc"] += dtanh
    gradients["dWax_enc"] += np.dot(dtanh, xt.T)
    gradients["dWaa_enc"] += np.dot(dtanh, h_prev.T)
    
    dh_prev = np.dot(Waa_enc.T, dtanh)
    return dh_prev, gradients

In [16]:
def encoder_backward(dh_last, enc_cache, parameters):
    x, h, caches = enc_cache
    
    gradients = {
        "dWax_enc": np.zeros_like(parameters["Wax_enc"]),
        "dWaa_enc": np.zeros_like(parameters["Waa_enc"]),
        "dba_enc": np.zeros_like(parameters["ba_enc"]),
    }
    
    dh = dh_last
    
    for t in reversed(range(len(caches))):
        dh, gradients = encoder_cell_backward(dh, caches[t], parameters, gradients)
    
    return gradients

In [17]:
def seq2seq_backward(tgt_output_indices, cache, parameters):
    enc_cache = cache["enc_cache"]
    dec_cache = cache["dec_cache"]
    
    dh_last, dec_grads = decoder_backward(tgt_output_indices, dec_cache, parameters)
    enc_grads = encoder_backward(dh_last, enc_cache, parameters)
    
    gradients = {}
    gradients.update(dec_grads)
    gradients.update(enc_grads)
    return gradients

In [18]:
def clip_seq2seq_gradients(gradients, max_value=5):
    for key in gradients:
        np.clip(gradients[key], -max_value, max_value, out=gradients[key])
    return gradients

In [19]:
def update_seq2seq_parameters(parameters, gradients, lr):
    for key in gradients:
        param_name = key[1:]  # dWax_enc -> Wax_enc
        parameters[param_name] -= lr * gradients[key]
    return parameters

In [20]:
def sentence_to_indices(sentence, word_to_idx):
    return [word_to_idx[w] for w in sentence.split()]

In [21]:
def prepare_training_example(src_sentence, tgt_sentence, src_word_to_idx, tgt_word_to_idx):
    src_indices = sentence_to_indices(src_sentence, src_word_to_idx)
    
    tgt_words = tgt_sentence.split()
    tgt_input_words = ["<START>"] + tgt_words
    tgt_output_words = tgt_words + ["<END>"]
    
    tgt_input_indices = [tgt_word_to_idx[w] for w in tgt_input_words]
    tgt_output_indices = [tgt_word_to_idx[w] for w in tgt_output_words]
    
    return src_indices, tgt_input_indices, tgt_output_indices

In [22]:
def get_top_k_probs(y_pred, idx_to_word, top_k=3):
    probs = y_pred.ravel()
    top_indices = np.argsort(probs)[::-1][:top_k]
    return [(idx_to_word[i], float(probs[i])) for i in top_indices]

In [23]:
def translate_with_probabilities(src_sentence, parameters,
                                 src_word_to_idx,
                                 tgt_word_to_idx,
                                 tgt_idx_to_word,
                                 n_a=32,
                                 max_len=10,
                                 top_k=3):
    
    src_indices = sentence_to_indices(src_sentence, src_word_to_idx)
    src_vocab_size = len(src_word_to_idx)
    tgt_vocab_size = len(tgt_word_to_idx)
    
    h0 = np.zeros((n_a, 1))
    h_enc, _ = encoder_forward(src_indices, h0, parameters, src_vocab_size)
    
    s_prev = h_enc
    current_idx = tgt_word_to_idx["<START>"]
    
    translated_words = []
    details = []
    
    for step in range(max_len):
        y_input = one_hot(current_idx, tgt_vocab_size)
        s_prev, y_pred, _ = decoder_cell_forward(y_input, s_prev, parameters)
        
        probs = y_pred.ravel()
        next_idx = np.argmax(probs)
        next_word = tgt_idx_to_word[next_idx]
        next_prob = float(probs[next_idx])
        
        top_candidates = get_top_k_probs(y_pred, tgt_idx_to_word, top_k=top_k)
        
        details.append({
            "step": step + 1,
            "decoder_input": tgt_idx_to_word[current_idx],
            "predicted_word": next_word,
            "predicted_prob": next_prob,
            "top_candidates": top_candidates
        })
        
        if next_word == "<END>":
            break
        
        translated_words.append(next_word)
        current_idx = next_idx
    
    return " ".join(translated_words), details

In [24]:
def train_seq2seq(pairs, n_a=32, lr=0.1, num_iterations=3000, print_every=300):
    src_sentences = [src for src, tgt in pairs]
    tgt_sentences = [tgt for src, tgt in pairs]
    
    src_vocab, src_word_to_idx, src_idx_to_word = build_vocab(src_sentences)
    tgt_vocab, tgt_word_to_idx, tgt_idx_to_word = build_vocab(
        tgt_sentences, add_special_tokens=["<START>", "<END>"]
    )
    
    src_vocab_size = len(src_vocab)
    tgt_vocab_size = len(tgt_vocab)
    
    parameters = initialize_seq2seq_parameters(
        n_a=n_a,
        n_x_src=src_vocab_size,
        n_x_tgt=tgt_vocab_size,
        n_y_tgt=tgt_vocab_size
    )
    
    h0 = np.zeros((n_a, 1))
    
    for i in range(num_iterations):
        total_loss = 0.0
        
        for src_sentence, tgt_sentence in pairs:
            src_indices, tgt_input_indices, tgt_output_indices = prepare_training_example(
                src_sentence, tgt_sentence, src_word_to_idx, tgt_word_to_idx
            )
            
            loss, cache = seq2seq_forward(
                src_indices, tgt_input_indices, tgt_output_indices,
                h0, parameters, src_vocab_size, tgt_vocab_size
            )
            
            gradients = seq2seq_backward(tgt_output_indices, cache, parameters)
            gradients = clip_seq2seq_gradients(gradients, 5)
            parameters = update_seq2seq_parameters(parameters, gradients, lr)
            
            total_loss += loss
        
        if i % print_every == 0:
            print(f"Iteration {i}, Total Loss: {total_loss:.4f}")
            test_translation, details = translate_with_probabilities(
                "i am happy",
                parameters,
                src_word_to_idx,
                tgt_word_to_idx,
                tgt_idx_to_word,
                n_a=n_a,
                max_len=6,
                top_k=3
            )
            print("Test: i am happy ->", test_translation)
            for d in details:
                print(d)
            print("-" * 60)
    
    return parameters, src_word_to_idx, src_idx_to_word, tgt_word_to_idx, tgt_idx_to_word

In [ ]:
parameters, src_word_to_idx, src_idx_to_word, tgt_word_to_idx, tgt_idx_to_word = train_seq2seq(
    pairs,
    n_a=64,
    lr=0.1,
    num_iterations=3000,
    print_every=300
)

In [26]:
test_sentences = [
    "i am happy",
    "i am sad",
    "you are kind",
    "he is tall",
    "she is smart"
]

for s in test_sentences:
    translation, details = translate_with_probabilities(
        s, parameters,
        src_word_to_idx,
        tgt_word_to_idx,
        tgt_idx_to_word,
        n_a=64,
        max_len=6,
        top_k=5
    )
    print("=" * 60)
    print("Input :", s)
    print("Output:", translation)
    print("Details:")
    for d in details:
        print(d)

Input : i am happy
Output: 我 很 开心
Details:
{'step': 1, 'decoder_input': '<START>', 'predicted_word': '我', 'predicted_prob': 0.9999744714589743, 'top_candidates': [('我', 0.9999744714589743), ('你', 1.3997922681833008e-05), ('他', 3.108655541764563e-06), ('很', 2.930725183914588e-06), ('聪明', 2.0004695137553954e-06)]}
{'step': 2, 'decoder_input': '我', 'predicted_word': '很', 'predicted_prob': 0.9999467842000839, 'top_candidates': [('很', 0.9999467842000839), ('善良', 2.510657267085579e-05), ('<END>', 1.08912083356271e-05), ('我', 7.378154948979869e-06), ('他', 5.9541850001356865e-06)]}
{'step': 3, 'decoder_input': '很', 'predicted_word': '开心', 'predicted_prob': 0.9999219593701472, 'top_candidates': [('开心', 0.9999219593701472), ('难过', 4.571993555608751e-05), ('高', 2.0546872703720903e-05), ('聪明', 6.812443794885536e-06), ('很', 2.626340441833416e-06)]}
{'step': 4, 'decoder_input': '开心', 'predicted_word': '<END>', 'predicted_prob': 0.9999903051225428, 'top_candidates': [('<END>', 0.9999903051225428), ('